# Historical cloud execution record

Saved outputs from the November 2 cloud run. This record references private `/workspace/SNOOPPI/work/review_20261102` artifacts. **It is not the portable entry point.** Open [01_results.ipynb](../01_results.ipynb) for a public-data rerun, or [02_retrain.ipynb](../02_retrain.ipynb) for opt-in retraining. Raw verification cannot run from the public repository alone.

# SNOOPPI cloud reproducibility and dry-run record — 2026-11-02

Fresh execution from repository commit fa2a8da. Synthetic tests establish implementation behavior, not biological performance. Original Drive CSVs and embedding caches are checked against the published hashes. Historical private splits and predictions were absent at session start. Settings available to the agent are the session instructions; no AGENTS.md was found. No independent confirmatory dataset or experimental mutation/interface annotations are supplied.

**Reconstruction qualification:** All three model partition CSVs exactly match the published SHA-256 values. Regenerated cluster mapping and protein-assignment files have different hashes; the old private mapping is unavailable, so byte identity or a row-order-only explanation is not claimed. Fresh mapping checks pass. The first stricter assertion failure is preserved under `attempts/`. Successful setup/check commands below reuse recorded receipts rather than rerunning them.

**Restart recovery:** On November 2, the cloud environment restarted after seeds 44 and 45 had completed. Seed 45 lacked an outer command receipt because the notebook kernel stopped; the training runner was invoked with `--resume` to verify its existing completion files. Its new command duration measures recovery checks, not original training. Seed 46 is a fresh run.

In [1]:
from pathlib import Path
import sys, os, json, hashlib, subprocess, time, platform
import pandas as pd
from IPython.display import display
ROOT=Path('/workspace/SNOOPPI')
WORK=ROOT/'work/review_20261102'
os.environ['PATH']=str(Path(sys.executable).parent)+os.pathsep+os.environ['PATH']
os.environ['MPLCONFIGDIR']=str(WORK/'mplconfig')
os.environ['OMP_NUM_THREADS']='2'
os.environ['OPENBLAS_NUM_THREADS']='2'
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def run(label,args,timeout=7200):
    receipt_path=WORK/'logs'/f'{label}.json'
    if receipt_path.exists():
        receipt=json.loads(receipt_path.read_text())
        print('Reusing completed command receipt:',receipt)
        return receipt['exit_code']
    args=list(args)
    if label.startswith('training_seed') and '--out' in args and Path(args[args.index('--out')+1]).exists() and '--resume' not in args: args.append('--resume')
    start=time.time()
    with (WORK/'logs'/f'{label}.log').open('w') as f:
        p=subprocess.run(args,cwd=ROOT,stdout=f,stderr=subprocess.STDOUT,timeout=timeout)
    receipt={'label':label,'command':args,'exit_code':p.returncode,'seconds':time.time()-start,'log_sha256':sha(WORK/'logs'/f'{label}.log')}
    (WORK/'logs'/f'{label}.json').write_text(json.dumps(receipt,indent=2))
    print(json.dumps(receipt,indent=2))
    print((WORK/'logs'/f'{label}.log').read_text()[-2500:])
    return p.returncode
print('Python:',sys.version)
print('Commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip())
(WORK/'requirements-frozen.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))


Python: 3.12.14 (main, Aug 25 2026, 14:00:49) [Clang 22.1.3 ]
Commit: fa2a8daab56b57768da737eda1cf3270cc766e69


2101

## Source verification
Six restored inputs must match the published source hashes before any fit.

In [2]:
audit=json.loads((ROOT/'experiments/cluster_disjoint/results/audit.json').read_text())
training=json.loads((ROOT/'experiments/cluster_disjoint/results/training_manifest.json').read_text())
records=[]
for name in ['train','validation','test','prott5','esm2','esmc']:
    path=WORK/'inputs'/(name+('.csv' if name in audit['sources'] else '.pt'))
    expected=audit['sources'][name] if name in audit['sources'] else training['caches'][name]['sha256']
    records.append(dict(name=name,bytes=path.stat().st_size,sha256=sha(path),matches_published=sha(path)==expected))
assert all(x['matches_published'] for x in records)
(WORK/'source_verification.json').write_text(json.dumps(records,indent=2))
display(pd.DataFrame(records))

,name,bytes,sha256,matches_published
0,train,44669229,56a9eaacf0af4322e7e318b54beaf1f113a60b82e6e2e3...,True
1,validation,5557519,d6d1177ec417c8bd6517da7618b17fcfb79273758abe8b...,True
2,test,5574700,ff23db8efbc30fc9f0021227f478bb506a3c6a7028470a...,True
3,prott5,69754503,93a72e4aaa6b6f0d9a3cbc4023453bd2747c62e4fe08fc...,True
4,esm2,82308919,ad72211f8c54b897592456a6e339a30233ae5f2cb8b121...,True
5,esmc,78936475,1267e1d40dc8c2d7444e9486b7c34c64ea2485505d6bde...,True


## Synthetic regression and notebook checks
These execute against synthetic fixtures. Any skips or failures remain visible in logs.

In [3]:
run('validate_notebooks',[sys.executable]+['scripts/validate_notebooks.py'])

Reusing completed command receipt: {'label': 'validate_notebooks', 'command': ['/workspace/.venvs/snooppi/bin/python', 'scripts/validate_notebooks.py'], 'exit_code': 0, 'seconds': 1.1673877239227295, 'log_sha256': 'c9dbc0945ad237d374c846529b05213a550cec250c220387068678624319f6f8'}


0

In [4]:
run('validate_drafts',[sys.executable]+['scripts/validate_drafts.py'])

Reusing completed command receipt: {'label': 'validate_drafts', 'command': ['/workspace/.venvs/snooppi/bin/python', 'scripts/validate_drafts.py'], 'exit_code': 0, 'seconds': 1.2188513278961182, 'log_sha256': '2e3d12976914433d0e5b76fbcff38db8d8546a071d053d94f0fe1e3c669cb649'}


0

In [5]:
run('regression',[sys.executable]+['-m', 'pytest', 'tests', 'experiments/cluster_disjoint', 'experiments/cluster_disjoint_v1', 'experiments/mmseqs2_benchmark', '--import-mode=importlib', '-q'])

Reusing completed command receipt: {'label': 'regression', 'command': ['/workspace/.venvs/snooppi/bin/python', '-m', 'pytest', 'tests', 'experiments/cluster_disjoint', 'experiments/cluster_disjoint_v1', 'experiments/mmseqs2_benchmark', '--import-mode=importlib', '-q'], 'exit_code': 2, 'seconds': 6.7841408252716064, 'log_sha256': '8ec143fdda7d41f0c1b8dcf7d5d2c52868691658129619f9460ebde9a2ec33ee'}


2

In [6]:
run('smoke_drafts',[sys.executable]+['scripts/smoke_drafts.py'])

Reusing completed command receipt: {'label': 'smoke_drafts', 'command': ['/workspace/.venvs/snooppi/bin/python', 'scripts/smoke_drafts.py'], 'exit_code': 0, 'seconds': 46.38506269454956, 'log_sha256': '98e218699c5c01bc673e7142256b8d3dfc52a9b80ad05b08dd80e408831cab10'}


0

In [7]:
run('smoke_mining',[sys.executable]+['scripts/smoke_mining.py'])

Reusing completed command receipt: {'label': 'smoke_mining', 'command': ['/workspace/.venvs/snooppi/bin/python', 'scripts/smoke_mining.py'], 'exit_code': 0, 'seconds': 14.333329200744629, 'log_sha256': 'bb327aafe70123a78bffd1ddace7dc14eacb5be99b7ad5368432fc20080e4fc0'}


0

## Reconstruct the historical benchmark
The MMseqs2 binary and resulting partitions are compared with the published hashes. Matching source data alone does not establish matching partitions.

In [8]:
binary=WORK/'bin/mmseqs/bin/mmseqs'
expected=json.loads((ROOT/'experiments/cluster_disjoint/results/clustering.json').read_text())
print('MMseqs2 version:',subprocess.check_output([str(binary),'version'],text=True).strip())
print('Binary matches historical:',sha(binary)==expected['binary_sha256'])
assert sha(binary)==expected['binary_sha256'], 'Exact historical binary required for this reconstruction'
assert run('prepare',[sys.executable,'experiments/cluster_disjoint/prepare.py','--inputs',str(WORK/'inputs'),'--out',str(WORK/'split'),'--mmseqs',str(binary)])==0
old=json.loads((ROOT/'experiments/cluster_disjoint/results/manifest.json').read_text())
checks={name:sha(WORK/'split'/name)==old['files'][name] for name in ['train.csv','validation.csv','test.csv','mmseqs2_clusters.csv']}
(WORK/'partition_reconstruction.json').write_text(json.dumps(checks,indent=2))
print(checks)
assert all(checks[n] for n in ['train.csv','validation.csv','test.csv']), 'Different model partitions'
print('Mapping hash differs; mapping byte identity is NOT claimed. Fresh mapping coverage and cross-partition leakage were checked by prepare.py and will be verified again.')


MMseqs2 version: 503743960fc57733a36951f5fa803cb3389af7e2
Binary matches historical: True
Reusing completed command receipt: {'label': 'prepare', 'command': ['/workspace/.venvs/snooppi/bin/python', 'experiments/cluster_disjoint/prepare.py', '--inputs', '/workspace/SNOOPPI/work/review_20261102/inputs', '--out', '/workspace/SNOOPPI/work/review_20261102/split', '--mmseqs', '/workspace/SNOOPPI/work/review_20261102/bin/mmseqs/bin/mmseqs'], 'exit_code': 0, 'seconds': 373.81304454803467, 'log_sha256': '305967985469568f693e7dc248dac95a1edec0d77c124e036957ff44659140b3'}


{'train.csv': True, 'validation.csv': True, 'test.csv': True, 'mmseqs2_clusters.csv': False}
Mapping hash differs; mapping byte identity is NOT claimed. Fresh mapping coverage and cross-partition leakage were checked by prepare.py and will be verified again.


## Matched real-data repetitions
Prespecified seeds 44, 45, 46; all 15 encoder/head/feature configurations per seed. No hyperparameter changes or test-driven selection. These are exploratory repeats on the existing benchmark, not an independent confirmatory test. CPU execution; frozen encoder caches are reused.

In [9]:
assert run('training_seed44',[sys.executable,'experiments/cluster_disjoint/train.py','--inputs',str(WORK/'inputs'),'--splits',str(WORK/'split'),'--out',str(WORK/'seed44'),'--seed','44'])==0

Reusing completed command receipt: {'label': 'training_seed44', 'command': ['/workspace/.venvs/snooppi/bin/python', 'experiments/cluster_disjoint/train.py', '--inputs', '/workspace/SNOOPPI/work/review_20261102/inputs', '--splits', '/workspace/SNOOPPI/work/review_20261102/split', '--out', '/workspace/SNOOPPI/work/review_20261102/seed44', '--seed', '44'], 'exit_code': 0, 'seconds': 729.3682684898376, 'log_sha256': '5b77be2cba906a2c08e55019c1e9e3f45b8228df44b7253a10084404de3c7ca2'}


In [10]:
assert run('training_seed45',[sys.executable,'experiments/cluster_disjoint/train.py','--inputs',str(WORK/'inputs'),'--splits',str(WORK/'split'),'--out',str(WORK/'seed45'),'--seed','45'])==0

{
  "label": "training_seed45",
  "command": [
    "/workspace/.venvs/snooppi/bin/python",
    "experiments/cluster_disjoint/train.py",
    "--inputs",
    "/workspace/SNOOPPI/work/review_20261102/inputs",
    "--splits",
    "/workspace/SNOOPPI/work/review_20261102/split",
    "--out",
    "/workspace/SNOOPPI/work/review_20261102/seed45",
    "--seed",
    "45",
    "--resume"
  ],
  "exit_code": 0,
  "seconds": 17.960699319839478,
  "log_sha256": "8c050a1e82034717cde9eb99429ee5e122a96d0217f04a8abde86d9dbf0dbeb3"
}
/workspace/SNOOPPI/experiments/cluster_disjoint/train.py:63: UserWarning: Legacy ProtT5 cache has no encoder revision/preprocessing metadata; dimension and coverage will be checked.
  c, meta = load_embedding_cache(args.inputs / f"{k}.pt", e)
/workspace/SNOOPPI/experiments/cluster_disjoint/train.py:105: UserWarning: Legacy ProtT5 cache has no encoder revision/preprocessing metadata; dimension and coverage will be checked.
  cache, _ = load_embedding_cache(args.inputs / f"{k

In [11]:
assert run('training_seed46',[sys.executable,'experiments/cluster_disjoint/train.py','--inputs',str(WORK/'inputs'),'--splits',str(WORK/'split'),'--out',str(WORK/'seed46'),'--seed','46'])==0

{
  "label": "training_seed46",
  "command": [
    "/workspace/.venvs/snooppi/bin/python",
    "experiments/cluster_disjoint/train.py",
    "--inputs",
    "/workspace/SNOOPPI/work/review_20261102/inputs",
    "--splits",
    "/workspace/SNOOPPI/work/review_20261102/split",
    "--out",
    "/workspace/SNOOPPI/work/review_20261102/seed46",
    "--seed",
    "46"
  ],
  "exit_code": 0,
  "seconds": 782.7272002696991,
  "log_sha256": "6ae2661fa9a6992b53d8bbbe82d97296ce301bcf44c9ca79aa27b0f8f6df1d4b"
}
/workspace/SNOOPPI/experiments/cluster_disjoint/train.py:63: UserWarning: Legacy ProtT5 cache has no encoder revision/preprocessing metadata; dimension and coverage will be checked.
  c, meta = load_embedding_cache(args.inputs / f"{k}.pt", e)
/workspace/SNOOPPI/experiments/cluster_disjoint/train.py:105: UserWarning: Legacy ProtT5 cache has no encoder revision/preprocessing metadata; dimension and coverage will be checked.
  cache, _ = load_embedding_cache(args.inputs / f"{k}.pt", e)
prott5_